In [ ]:
''' Verifies qubit_mapping by reproducing the first-depth single-qubit gates
from circuit_n12_m14_s0_e0_pEFGH_snapped.json using MBQC tetron gates,
then comparing per-qubit P(0) against a direct Qiskit reference circuit.'''
import sys, json
from pathlib import Path
from collections import defaultdict

import numpy as np
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.compiler import transpile

repo_root = Path(__file__).resolve().parent
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src" / "tetron"))

from src.tetron.qubit_mapping import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    GRID_TO_LOGICAL,
    LOGICAL_TO_TETRON_SITE,
    LOGICAL_TO_SQ_ANCILLA_SITE,
)
from single_qubit_gate import TetronSingleQubitGates

In [ ]:
# ── 1. Load first moment ──────────────────────────────────────────────────────

json_path = repo_root / "src/circuits/processed/circuit_n12_m14_s0_e0_pEFGH_snapped.json"
with open(json_path) as f:
    circuit_data = json.load(f)

first_moment = circuit_data["moments"][0]
assert first_moment["depth"] == 0

LOGICAL_TO_GRID = {v: k for k, v in GRID_TO_LOGICAL.items()}

print("First moment gates:")
for g in first_moment["gates"]:
    label = g["qubits"][0]
    row, col = LOGICAL_TO_GRID[label]
    d_wire = grid_to_qiskit_index(row, col)
    a_wire = grid_to_sq_ancilla_index(row, col)
    print(f"  {g['gate']:8s}  label={label:2d}  Cirq({row},{col})"
          f"  -> data q[{d_wire:2d}] (T{LOGICAL_TO_TETRON_SITE[label]})"
          f"   ancilla q[{a_wire:2d}] (T{LOGICAL_TO_SQ_ANCILLA_SITE[label]})")

In [ ]:
# ── 2. Build MBQC circuit ─────────────────────────────────────────────────────

GATE_FN = {
    "sqrtX": TetronSingleQubitGates.gate_sqrt_X,
    "sqrtY": TetronSingleQubitGates.gate_sqrt_Y,
    "sqrtW": TetronSingleQubitGates.gate_sqrt_W,
}
GATE_BITS = {"sqrtX": 4, "sqrtY": 9, "sqrtW": 4}

N_PHYS  = 24
n_cbits = sum(GATE_BITS[g["gate"]] for g in first_moment["gates"])

creg     = ClassicalRegister(n_cbits, name="c_int")
meas_reg = ClassicalRegister(12,      name="meas")
qc_mbqc  = QuantumCircuit(N_PHYS, name="mbqc_moment0")
qc_mbqc.add_register(creg)
qc_mbqc.add_register(meas_reg)

bit_ptr        = 0
data_wire_order = []

for g in first_moment["gates"]:
    gate_name = g["gate"]
    label     = g["qubits"][0]
    row, col  = LOGICAL_TO_GRID[label]
    d_wire    = grid_to_qiskit_index(row, col)
    a_wire    = grid_to_sq_ancilla_index(row, col)

    qc_mbqc.h(a_wire)
    GATE_FN[gate_name](qc_mbqc, a_wire, d_wire, creg, start_idx=bit_ptr)

    data_wire_order.append((label, d_wire))
    bit_ptr += GATE_BITS[gate_name]

data_wires_sorted = sorted(data_wire_order, key=lambda x: x[1])
for i, (label, d_wire) in enumerate(data_wires_sorted):
    qc_mbqc.measure(d_wire, meas_reg[i])

print(f"\nMBQC circuit: {qc_mbqc.num_qubits} qubits, depth {qc_mbqc.depth()}, "
      f"{bit_ptr} intermediate + 12 meas classical bits")

In [ ]:
# ── 3. Build direct reference circuit ────────────────────────────────────────

def apply_direct(qc, gate_name, data_wire):
    if gate_name == "sqrtX":
        qc.sx(data_wire)
    elif gate_name == "sqrtY":
        qc.s(data_wire); qc.s(data_wire); qc.h(data_wire)
    elif gate_name == "sqrtW":
        qc.tdg(data_wire); qc.sx(data_wire); qc.t(data_wire)

meas_ref  = ClassicalRegister(12, name="meas")
qc_direct = QuantumCircuit(N_PHYS, name="direct_moment0")
qc_direct.add_register(meas_ref)

for g in first_moment["gates"]:
    gate_name = g["gate"]
    label     = g["qubits"][0]
    row, col  = LOGICAL_TO_GRID[label]
    d_wire    = grid_to_qiskit_index(row, col)
    apply_direct(qc_direct, gate_name, d_wire)

for i, (label, d_wire) in enumerate(data_wires_sorted):
    qc_direct.measure(d_wire, meas_ref[i])

print(f"Direct circuit: {qc_direct.num_qubits} qubits, depth {qc_direct.depth()}")

In [ ]:
# ── 4. Simulate and compare ───────────────────────────────────────────────────

SHOTS = 4096
sim   = AerSimulator()

print("\nRunning MBQC simulation...")
res_mbqc   = sim.run(transpile(qc_mbqc,   sim, optimization_level=0), shots=SHOTS).result()
print("Running direct simulation...")
res_direct = sim.run(transpile(qc_direct, sim, optimization_level=0), shots=SHOTS).result()

counts_mbqc   = res_mbqc.get_counts()
counts_direct = res_direct.get_counts()

# Marginalise MBQC over c_int; meas_reg is the leftmost token in Qiskit output
marg_mbqc = defaultdict(int)
for bs, cnt in counts_mbqc.items():
    marg_mbqc[bs.split(" ")[0]] += cnt
marg_mbqc = dict(marg_mbqc)

def marginal_p0(counts, qubit_pos, n_bits, shots):
    count0 = sum(cnt for bs, cnt in counts.items()
                 if len(bs) >= n_bits and bs[-(qubit_pos + 1)] == "0")
    return count0 / shots

n_data = 12
print(f"\n  {'label':>6}  {'data wire':>10}  {'P(0) MBQC':>11}  {'P(0) direct':>12}  {'diff':>7}")
print("  " + "-" * 58)
max_diff = 0.0
for i, (label, d_wire) in enumerate(data_wires_sorted):
    p0_m = marginal_p0(marg_mbqc,   i, n_data, SHOTS)
    p0_d = marginal_p0(counts_direct, i, n_data, SHOTS)
    diff = abs(p0_m - p0_d)
    max_diff = max(max_diff, diff)
    gate_applied = next(g["gate"] for g in first_moment["gates"] if g["qubits"][0] == label)
    print(f"  L{label:>2d} ({gate_applied:6s})  q[{d_wire:2d}]       {p0_m:>9.3f}    {p0_d:>10.3f}   {diff:>6.3f}")

print("  " + "-" * 58)
tol = 3 / SHOTS**0.5   # ~3 sigma
print(f"  Max |diff| = {max_diff:.3f}   tolerance = {tol:.3f}  (3σ shot noise)")
print(f"\n  RESULT: {'PASSED ✓' if max_diff < tol else 'FAILED ✗'}")


In [ ]:
# ── 5. Per-gate unitary check from |0⟩ ───────────────────────────────────────

print("\n── Per-gate check from |0⟩ (all three gates should give P(0)=0.5) ──")
SHOTS_SINGLE = 2048
print(f"{'Gate':8s}  {'Label':6s}  {'P(0) MBQC':>10}  {'Expected':>9}  {'OK?':>5}")
print("-" * 48)

for g in first_moment["gates"]:
    gate_name = g["gate"]
    label     = g["qubits"][0]
    row, col  = LOGICAL_TO_GRID[label]
    d_wire    = grid_to_qiskit_index(row, col)
    a_wire    = grid_to_sq_ancilla_index(row, col)
    n_bits    = GATE_BITS[gate_name]

    cr_i = ClassicalRegister(n_bits, name="c")
    cr_m = ClassicalRegister(1,      name="m")
    qc_i = QuantumCircuit(N_PHYS)
    qc_i.add_register(cr_i)
    qc_i.add_register(cr_m)

    qc_i.h(a_wire)
    GATE_FN[gate_name](qc_i, a_wire, d_wire, cr_i, start_idx=0)
    qc_i.measure(d_wire, cr_m[0])

    res_i = sim.run(transpile(qc_i, sim, optimization_level=0),
                    shots=SHOTS_SINGLE).result().get_counts()

    # cr_m is the leftmost token
    cnt0 = sum(cnt for bs, cnt in res_i.items() if bs.split(" ")[0] == "0")
    p0   = cnt0 / SHOTS_SINGLE
    ok   = "OK" if abs(p0 - 0.5) < 0.05 else "FAIL"
    print(f"{gate_name:8s}  L{label:>2d}      {p0:>10.3f}  {0.5:>9.3f}  {ok:>5}")
